# REV6 run 1 of 2: multi-cell in-domain (the core run)

REV6 adds to the REV5 multi-cell experiment, without changing REV5:
1. **Post-processing fitted out of sample**: a development model (1996-2016) forecasts 2017-2020; quantile mapping (QM),
   variance inflation (VI) and an alert-threshold shift (THR) are fitted there and applied to the 2021-2025 test forecasts.
2. **Onset verification** (issue day below the alert level) and **episode-onset verification** (±7 days).
3. **Block bootstrap** 95% intervals for CSI and for the CSI difference against persistence.
4. **Transformer with positional encoding**.

**Time:** about 5 h on GPU (two fits per lead). **Settings:** Accelerator GPU; Input: the `ems-tcn-full` dataset (same as REV5).
Run cells 0-4 first (a few minutes). If the smoke test passes, use **Save Version -> Save & Run All (Commit)**.
Do not keep an interactive session running at the same time (it uses a second GPU session).
At the end, download **`results_rev6_indomain.zip`** from the Output tab and send it back.

In [ ]:
# 0 | Inputs
import glob
for lab, pat in {'pipeline v2': '/kaggle/input/**/tcn_drought_pipeline_v2.py', 'cells.csv': '/kaggle/input/**/cells.csv',
                 'cell_*.csv': '/kaggle/input/**/cell_*.csv'}.items():
    f = glob.glob(pat, recursive=True); print(f"{'OK ' if f else '-- '} {lab:12s} {len(f)} found")


In [ ]:
# 1 | Setup and GPU check
import os, sys, shutil, glob, time
src = glob.glob('/kaggle/input/**/tcn_drought_pipeline_v2.py', recursive=True)[0]
shutil.copy(src, '/kaggle/working/'); os.chdir('/kaggle/working'); sys.path.insert(0, '/kaggle/working')
DATA_DIR = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/cells.csv', recursive=True)
            if len(glob.glob(os.path.join(os.path.dirname(f), 'cell_*.csv'))) >= 10][0]
OUT_DIR = '/kaggle/working/results_rev6'; os.makedirs(OUT_DIR, exist_ok=True)
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU'); print('TF', tf.__version__, '| GPUs:', gpus, '| DATA_DIR:', DATA_DIR)
assert gpus, 'STOP: no GPU visible to TensorFlow (Settings -> Accelerator -> GPU).'


In [ ]:
%%writefile tcn_drought_multicell.py
"""tcn_drought_multicell.py  --  REV5 (REV4 + per-cell chronological validation for the neural models; see CHANGES_REV5.md)


================================================================================
MULTI-CELL DROUGHT FORECASTING  (transfer / spatial generalization)  -- REV4
--------------------------------------------------------------------------------
Pooled, cross-cell analysis (NOT ten independent single-cell models):
  * in-domain: ONE pooled model trained on windows from ALL cells (with static
    covariates), tested on each cell's 2021-2025;                                [2.16]
  * transfer (leave-one-cell-out): pooled model trained on all cells except cell
    k, tested on cell k (never seen);                                            [2.16]
  * static covariates (lat, lon, elev, mean annual P, mean dry-season JJAS P,
    mean soil moisture, aridity P/PET), z-scored across the training cells;
  * a REGIONAL early-warning rule: alert when >= regional_frac of cells cross
    the threshold; POD/FAR/CSI on the regional alert series.                     [2.12]
All physics / index / model / DSS code is imported from tcn_drought_pipeline_v2.

REV4 (release) folds into this module every notebook-level patch used to produce the
published multi-cell results, so that a plain run reproduces them without extra cells:
  * Random Forest with max_features="sqrt" (rf_max_features);
  * transfer Random Forest with 100 trees (transfer_rf_trees); in-domain keeps rf_trees;
  * warning verification at levels D0-D3 (levels) in the in-domain phase;
  * warning verification in the transfer phase (per held-out cell, OR and AND, D0-D3),
    merged into dss_transfer_percell.csv and pooled into dss_transfer_multicell.csv;
  * Climatology continuous metrics in the transfer phase;
  * regional alert scored at several aggregation fractions (regional_fracs), written to
    dss_regional_thresholds.csv; dss_regional.csv keeps the primary fraction (regional_frac);
  * optional export of daily observed and forecast SPEI/SSI per cell (save_values), so any
    further fraction, level or rule can be scored without retraining.

SHARDING (run the long job in short, resumable pieces with IDENTICAL results)
----------------------------------------------------------------------------
The transfer loop is independent per held-out cell, and in-domain is independent
per horizon, so the run can be split and merged exactly:
  CONFIG keys:
    phase         : "all" (default) | "indomain" | "transfer" | "merge"
    only_held     : None | "<cell_id>"  (transfer: run just this held-out cell)
    only_horizons : None | [h, ...]     (restrict horizons for this job)
  Shards write partial CSVs under <out_dir>/parts/ ; phase="merge" concatenates
  them into the final metrics_indomain.csv / metrics_transfer.csv / dss_*.csv /
  skill_map.csv. A plain run (phase="all", no only_*) behaves exactly as before.

Inputs: data/cells.csv + data/cell_<id>.csv (with WS2M) from the downloader.
TensorFlow is optional - RF + baselines alone run the whole thing.
Usage:  python tcn_drought_multicell.py
================================================================================
"""
import os, glob
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
import tcn_drought_pipeline_v2 as P          # canonical functions (single source of truth)

CONFIG_MC = dict(
    data_dir="data", out_dir="results_mc",
    train_end="2016-12-31", val_end="2020-12-31",         # 3-way split; calib = train+val
    test_start="2021-01-01", test_end="2025-12-31",
    targets=["GWETROOT", "SPEI_1"],
    static_feats=["lat", "lon", "elev", "clim_precip", "clim_dryP", "clim_gwet", "aridity"],
    lookback=90, horizons=[1, 7, 14, 30], seeds=[0, 1, 2],
    epochs=60, batch_size=64, rf_trees=200, spei_clip=3.0,
    spei_scale=30, doy_halfwin=15, wind_var="WS2M", wind_default=2.0,
    transfer_models=["TCN", "RandomForest"],
    map_horizon=14, map_target="GWETROOT",
    regional_frac=0.30,                                   # >=30% of cells -> regional alert
    regional_fracs=(0.20, 0.30, 0.40, 0.50),              # sensitivity of the regional alert
    levels=(1, 2, 3, 4),                                  # verified drought levels D0..D3
    rf_max_features="sqrt",                               # predictors considered per split
    transfer_rf_trees=100,                                # RF trees in the transfer phase
    save_values=True,                                     # daily SPEI/SSI export per cell
    val_frac=0.10,                                        # REV5: last 10% of EACH cell's windows for early stopping
    # ---- sharding controls ----
    phase="all", only_held=None, only_horizons=None,
)

def _parts(cfg):
    p = os.path.join(cfg["out_dir"], "parts"); os.makedirs(p, exist_ok=True); return p
def _horizons(cfg):
    return cfg["only_horizons"] if cfg.get("only_horizons") else cfg["horizons"]
def _htag(cfg):
    return "h" + "_".join(str(h) for h in _horizons(cfg))

# ------------------------------- data loading ---------------------------------
def load_cells(cfg):
    meta = pd.read_csv(os.path.join(cfg["data_dir"], "cells.csv"))
    raw = {}
    for _, r in meta.iterrows():
        path = os.path.join(cfg["data_dir"], f"cell_{r['cell_id']}.csv")
        if not os.path.exists(path):
            print(f"  [skip] missing {path}"); continue
        df = pd.read_csv(path, parse_dates=["DATE"]).set_index("DATE").sort_index()
        raw[r["cell_id"]] = df
    return meta, raw

def process_cell(rdf, lat, elev, cfg):
    df = rdf.copy().interpolate("time").ffill().bfill()
    df = P.calculate_pet(df, lat, elev, cfg["wind_var"], cfg["wind_default"])
    df = P.calculate_spei(df, cfg["val_end"], cfg["spei_clip"], cfg["spei_scale"], cfg["doy_halfwin"])
    cm = df.index <= pd.Timestamp(cfg["val_end"])
    df["SSI"] = P.standardized_soil_moisture_index(df["GWETROOT"], cm, cfg["spei_clip"], cfg["doy_halfwin"])
    return df

def static_raw(df, meta_row, cfg):
    cal = df.loc[:cfg["val_end"]]
    p_ann, pet_ann = float(cal["PRECTOTCORR"].mean()), float(cal["PET"].mean())
    jjas = float(cal[cal.index.month.isin([6, 7, 8, 9])]["PRECTOTCORR"].mean())
    return dict(lat=float(meta_row["lat"]), lon=float(meta_row["lon"]), elev=float(meta_row["elev"]),
                clim_precip=p_ann, clim_dryP=jjas, clim_gwet=float(cal["GWETROOT"].mean()),
                aridity=(p_ann / pet_ann if pet_ann else np.nan))

def prepare(cfg):
    meta, raw = load_cells(cfg)
    has_ws = len(raw) > 0 and all("WS2M" in rdf.columns for rdf in raw.values())
    cfg = dict(cfg); cfg["features"] = P.BASE_FEATURES + (["WS2M"] if has_ws else [])
    print(f"Features ({'with' if has_ws else 'no'} WS2M): {cfg['features']}")
    cells = {}
    for cid, rdf in raw.items():
        mr = meta.loc[meta.cell_id == cid].iloc[0]
        df = process_cell(rdf, float(mr["lat"]), float(mr["elev"]), cfg)
        d = P.split_scale(df, cfg)
        cm = df.index <= pd.Timestamp(cfg["val_end"])
        cells[cid] = dict(df=df, d=d, ssi_clim=P.fit_ssi_climatology(df["GWETROOT"], cm, cfg["doy_halfwin"]),
                          static=static_raw(df, mr, cfg), meta=mr)
    return meta, cells, cfg

# --------------------- static covariates (fold-specific z-score) --------------
def zscore_static(cells, train_ids, cfg):
    F = cfg["static_feats"]
    M = np.array([[cells[c]["static"][f] for f in F] for c in train_ids])
    mu, sd = M.mean(0), M.std(0); sd[sd == 0] = 1.0
    return {c: (np.array([cells[c]["static"][f] for f in F]) - mu) / sd for c in cells}

def cell_windows(cells, cid, zstat, h, cfg, split):
    d = cells[cid]["d"]; L = cfg["lookback"]
    Xd, Y = (d["Xtr"], d["Ytr"]) if split == "train" else (d["Xte"], d["Yte"])
    Xfull = np.hstack([Xd, np.tile(zstat[cid], (len(Xd), 1))])
    return P.build_windows(Xfull, Y, L, h)

# ------------------------------ pooled fit/predict ----------------------------
def split_per_cell(Xs, Ys, frac):
    """REV5: chronological split of EACH cell's calibration windows. The first (1 - frac) of every cell's windows are
    used for fitting the neural models and the last frac for early stopping, so that every cell contributes to both.
    (REV4 passed the pooled array to Keras with validation_split, which held out the tail of the concatenation,
    i.e. almost all of the last cell.)"""
    Xf, Yf, Xv, Yv = [], [], [], []
    for X, Y in zip(Xs, Ys):
        k = int(round(len(X) * (1 - frac)))
        Xf.append(X[:k]); Yf.append(Y[:k]); Xv.append(X[k:]); Yv.append(Y[k:])
    return np.concatenate(Xf), np.concatenate(Yf), np.concatenate(Xv), np.concatenate(Yv)

def train_keras_val(builder, Xfit, Yfit, Xval, Yval, in_shape, cfg, seed):
    """Identical to tcn_drought_pipeline_v2.train_keras except that the validation set is passed explicitly."""
    P._set_seed(seed)
    model = builder(in_shape)
    es = P.EarlyStopping(patience=8, restore_best_weights=True, monitor="val_loss")
    model.fit(Xfit, Yfit, validation_data=(Xval, Yval), epochs=cfg["epochs"],
              batch_size=cfg["batch_size"], verbose=0, callbacks=[es])
    return model

def fit_pool(Xtr, Ytr, cfg, models, n_trees=None, keras_split=None):
    """Random Forest: all calibration windows (unchanged from REV4).
    Neural models: keras_split = (Xfit, Yfit, Xval, Yval) from split_per_cell (REV5)."""
    fitted = {}
    if "RandomForest" in models:
        fitted["RandomForest"] = []
        for s in cfg["seeds"]:
            rf = RandomForestRegressor(n_estimators=n_trees or cfg["rf_trees"],
                                       max_features=cfg.get("rf_max_features", "sqrt"),
                                       random_state=s, n_jobs=-1)
            rf.fit(Xtr.reshape(len(Xtr), -1), Ytr); fitted["RandomForest"].append(("rf", rf))
    if P.HAS_TF:
        in_shape = (cfg["lookback"], Xtr.shape[2])
        bmap = {"TCN": P.build_tcn, "LSTM": P.build_lstm, "Transformer": P.build_transformer}
        for mname in models:
            if mname not in bmap:
                continue
            if keras_split is None:
                raise ValueError("REV5 requires keras_split (per-cell validation) for the neural models")
            Xf, Yf, Xv, Yv = keras_split
            fitted[mname] = [("keras", train_keras_val(bmap[mname], Xf, Yf, Xv, Yv, in_shape, cfg, s))
                             for s in cfg["seeds"]]
    return fitted

def predict_pool(fitted, Xte):
    out = {}
    for mname, models in fitted.items():
        ps = [m.predict(Xte.reshape(len(Xte), -1)) if kind == "rf" else m.predict(Xte, verbose=0)
              for kind, m in models]
        out[mname] = np.mean(ps, axis=0)
    return out

# ------------------------------- in-domain run --------------------------------
def _alert_scores(o, f):
    hits = int((f & o).sum()); fa = int((f & ~o).sum()); miss = int((~f & o).sum()); cn = int((~f & ~o).sum())
    return dict(POD=hits / (hits + miss) if (hits + miss) else np.nan,
                FAR=fa / (hits + fa) if (hits + fa) else np.nan,
                CSI=hits / (hits + miss + fa) if (hits + miss + fa) else np.nan,
                base_rate=(hits + miss) / len(o) if len(o) else np.nan,
                hits=hits, false_alarms=fa, misses=miss, correct_neg=cn)

def regional_rows(status_obs, status_fc, h, fracs, levels):
    """Regional alert: fires when >= frac of cells are at or above a level (OR rule).
    status_obs: DataFrame date x cell of integer severity; status_fc: {model: same}."""
    rows = []
    for frac in fracs:
        for level in levels:
            OBS = (status_obs >= level).mean(axis=1) >= frac
            for m, sf in status_fc.items():
                FC = (sf >= level).mean(axis=1) >= frac
                a, b = OBS.align(FC, join="inner")
                rows.append(dict(horizon=h, model=m, logic="or", level=P.LEVEL_NAMES[level], threshold=frac,
                                 **_alert_scores(a.values, b.values)))
    return rows

def run_indomain(cells, cfg):
    ids = list(cells.keys())
    z = zscore_static(cells, ids, cfg)
    gi, si = cfg["targets"].index("GWETROOT"), cfg["targets"].index("SPEI_1")
    learned = ["RandomForest"] + (["TCN", "LSTM", "Transformer"] if P.HAS_TF else [])
    levels = tuple(cfg.get("levels", (1, 2)))
    fracs = tuple(cfg.get("regional_fracs", (cfg["regional_frac"],)))
    rows, dss_pool, reg_rows = [], [], []

    for h in _horizons(cfg):
        Xtr, Ytr = [], []
        for cid in ids:
            Xw, Yw, _ = cell_windows(cells, cid, z, h, cfg, "train"); Xtr.append(Xw); Ytr.append(Yw)
        fitted = fit_pool(np.concatenate(Xtr), np.concatenate(Ytr), cfg, learned,
                          keras_split=split_per_cell(Xtr, Ytr, cfg["val_frac"]))

        models_all = ["Persistence", "Climatology"] + learned
        pool_obs = {"spei": [], "ssi": []}
        pool_fc = {m: {"spei": [], "ssi": []} for m in models_all}
        st_obs, st_fc, values = {}, {m: {} for m in models_all}, []

        for cid in ids:
            Xte, Yte, Ylast = cell_windows(cells, cid, z, h, cfg, "test")
            d = cells[cid]["d"]; inv = d["sY"].inverse_transform
            Yte_o, Ylast_o = inv(Yte), inv(Ylast)
            tdates = d["te_index"][cfg["lookback"] + h - 1: cfg["lookback"] + h - 1 + len(Yte)]
            tdoy = np.array([dt.dayofyear for dt in tdates]); clim = cells[cid]["ssi_clim"]

            preds = {"Persistence": {gi: Ylast_o[:, gi], si: Ylast_o[:, si]},
                     "Climatology": {gi: P.climatology_pred(d["tr_index"], d["tr_raw"][:, gi], tdates),
                                     si: P.climatology_pred(d["tr_index"], d["tr_raw"][:, si], tdates)}}
            scaled = predict_pool(fitted, Xte)
            for m in learned:
                po = inv(scaled[m]); preds[m] = {gi: po[:, gi], si: po[:, si]}

            for m, pr in preds.items():
                for ti, tname in [(gi, "GWETROOT"), (si, "SPEI_1")]:
                    rows.append(dict(cell_id=cid, horizon=h, target=tname, model=m,
                                     **P.all_metrics(Yte_o[:, ti], pr[ti], Ylast_o[:, ti])))

            idx = pd.DatetimeIndex(tdates)
            oss = P.apply_ssi(Yte_o[:, gi], tdoy, clim, cfg["spei_clip"]); osp = Yte_o[:, si]
            pool_obs["spei"].append(osp); pool_obs["ssi"].append(oss)
            st_obs[cid] = pd.Series(P.status_series(osp, oss, "or"), index=idx)
            vrow = pd.DataFrame({"date": idx, "cell_id": cid, "obs_spei": osp, "obs_ssi": oss})
            for m, pr in preds.items():
                fss = P.apply_ssi(pr[gi], tdoy, clim, cfg["spei_clip"]); fsp = pr[si]
                pool_fc[m]["spei"].append(fsp); pool_fc[m]["ssi"].append(fss)
                st_fc[m][cid] = pd.Series(P.status_series(fsp, fss, "or"), index=idx)
                vrow[f"{m}_spei"] = fsp; vrow[f"{m}_ssi"] = fss
            values.append(vrow)

        if cfg.get("save_values", False):
            sdir = os.path.join(cfg["out_dir"], "status"); os.makedirs(sdir, exist_ok=True)
            pd.concat(values).to_csv(os.path.join(sdir, f"values__h{h}.csv.gz"), index=False)

        OS, OG = np.concatenate(pool_obs["spei"]), np.concatenate(pool_obs["ssi"])
        for m in models_all:
            FS, FG = np.concatenate(pool_fc[m]["spei"]), np.concatenate(pool_fc[m]["ssi"])
            for logic in ("or", "and"):
                for level in levels:
                    dss_pool.append(dict(horizon=h, model=m, logic=logic, level=P.LEVEL_NAMES[level],
                                         **P.validate_forecast_dss(OS, OG, FS, FG, logic, level)))
        reg_rows += regional_rows(pd.DataFrame(st_obs), {m: pd.DataFrame(st_fc[m]) for m in models_all},
                                  h, fracs, levels)
    return pd.DataFrame(rows), pd.DataFrame(dss_pool), pd.DataFrame(reg_rows)

# ----------------------- leave-one-cell-out transfer --------------------------
def run_transfer(cells, cfg):
    """Leave-one-cell-out transfer. Returns (continuous metrics, per-cell warning verification)."""
    ids = list(cells.keys())
    held_list = [cfg["only_held"]] if cfg.get("only_held") else ids
    for hc in held_list:
        if hc not in ids:
            raise ValueError(f"only_held='{hc}' not among cells {ids}")
    gi, si = cfg["targets"].index("GWETROOT"), cfg["targets"].index("SPEI_1")
    models = cfg["transfer_models"]; levels = tuple(cfg.get("levels", (1, 2)))
    rows, dss_rows = [], []
    for held in held_list:
        train_ids = [c for c in ids if c != held]
        z = zscore_static(cells, train_ids, cfg)
        for h in _horizons(cfg):
            Xtr, Ytr = [], []
            for cid in train_ids:
                Xw, Yw, _ = cell_windows(cells, cid, z, h, cfg, "train"); Xtr.append(Xw); Ytr.append(Yw)
            fitted = fit_pool(np.concatenate(Xtr), np.concatenate(Ytr), cfg, models,
                              n_trees=cfg.get("transfer_rf_trees"),
                              keras_split=split_per_cell(Xtr, Ytr, cfg["val_frac"]))
            Xte, Yte, Ylast = cell_windows(cells, held, z, h, cfg, "test")
            d = cells[held]["d"]; inv = d["sY"].inverse_transform
            Yte_o, Ylast_o = inv(Yte), inv(Ylast)
            scaled = predict_pool(fitted, Xte)
            tdates = d["te_index"][cfg["lookback"] + h - 1: cfg["lookback"] + h - 1 + len(Yte)]
            tdoy = np.array([dt.dayofyear for dt in tdates]); clim = cells[held]["ssi_clim"]

            preds = {}
            for m in models:
                if m in scaled:
                    po = inv(scaled[m]); preds[m] = {gi: po[:, gi], si: po[:, si]}
            preds["Persistence"] = {gi: Ylast_o[:, gi], si: Ylast_o[:, si]}
            preds["Climatology"] = {gi: P.climatology_pred(d["tr_index"], d["tr_raw"][:, gi], tdates),
                                    si: P.climatology_pred(d["tr_index"], d["tr_raw"][:, si], tdates)}
            for m, pr in preds.items():
                for ti, tname in [(gi, "GWETROOT"), (si, "SPEI_1")]:
                    rows.append(dict(held_out=held, horizon=h, target=tname, model=m, regime="transfer",
                                     **P.all_metrics(Yte_o[:, ti], pr[ti], Ylast_o[:, ti])))
            oss = P.apply_ssi(Yte_o[:, gi], tdoy, clim, cfg["spei_clip"]); osp = Yte_o[:, si]
            for m, pr in preds.items():
                fss = P.apply_ssi(pr[gi], tdoy, clim, cfg["spei_clip"]); fsp = pr[si]
                for logic in ("or", "and"):
                    for level in levels:
                        dss_rows.append(dict(held_out=held, horizon=h, model=m, logic=logic,
                                             level=P.LEVEL_NAMES[level], regime="transfer",
                                             **P.validate_forecast_dss(osp, oss, fsp, fss, logic, level)))
    return pd.DataFrame(rows), pd.DataFrame(dss_rows)

def pool_transfer_dss(per_cell):
    """Pool per-held-out-cell verification by summing contingency counts (not averaging ratios)."""
    k = ["horizon", "model", "logic", "level"]
    p = per_cell.groupby(k)[["hits", "false_alarms", "misses", "correct_neg"]].sum().reset_index()
    n = p.hits + p.false_alarms + p.misses + p.correct_neg
    p["POD"] = p.hits / (p.hits + p.misses); p["FAR"] = p.false_alarms / (p.hits + p.false_alarms)
    p["CSI"] = p.hits / (p.hits + p.misses + p.false_alarms); p["base_rate"] = (p.hits + p.misses) / n
    p["freq_bias"] = (p.hits + p.false_alarms) / (p.hits + p.misses)
    return p

# ------------------------------- skill map / merge ----------------------------
def build_skill_map(meta, ind, tr, cfg):
    primary = "TCN" if P.HAS_TF else "RandomForest"
    mh = cfg["map_horizon"] if cfg["map_horizon"] in cfg["horizons"] else cfg["horizons"][0]
    mt = cfg["map_target"]
    i_s = (ind[(ind.horizon == mh) & (ind.target == mt) & (ind.model == primary)]
           [["cell_id", "SS_vs_persist"]].rename(columns={"SS_vs_persist": "SS_indomain"}))
    t_s = (tr[(tr.horizon == mh) & (tr.target == mt) & (tr.model == primary)]
           [["held_out", "SS_vs_persist"]].rename(columns={"held_out": "cell_id",
                                                            "SS_vs_persist": "SS_transfer"}))
    return (meta[["cell_id", "name", "lat", "lon"]].merge(i_s, on="cell_id", how="left")
            .merge(t_s, on="cell_id", how="left"))

def _concat(prefix, cfg):
    fs = sorted(glob.glob(os.path.join(cfg["out_dir"], "parts", prefix + "*.csv")))
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()

def merge_shards(cfg):
    meta = pd.read_csv(os.path.join(cfg["data_dir"], "cells.csv"))
    ind = _concat("indomain__", cfg)
    if ind.empty and os.path.exists(f"{cfg['out_dir']}/metrics_indomain.csv"):
        ind = pd.read_csv(f"{cfg['out_dir']}/metrics_indomain.csv")
    dss = _concat("dssmc__", cfg)
    reg = _concat("dssreg__", cfg)
    dtr = _concat("dsstr__", cfg)
    tr = _concat("transfer__", cfg)
    if tr.empty and os.path.exists(f"{cfg['out_dir']}/metrics_transfer.csv"):
        tr = pd.read_csv(f"{cfg['out_dir']}/metrics_transfer.csv")
    # de-duplicate (in case shards overlap)
    if not ind.empty: ind = ind.drop_duplicates(["cell_id", "horizon", "target", "model"])
    if not tr.empty:  tr  = tr.drop_duplicates(["held_out", "horizon", "target", "model"])
    if not dss.empty: dss = dss.drop_duplicates(["horizon", "model", "logic", "level"])
    if not reg.empty:
        if "threshold" not in reg.columns: reg["threshold"] = cfg["regional_frac"]
        reg = reg.drop_duplicates(["horizon", "model", "level", "threshold"])
    if not dtr.empty: dtr = dtr.drop_duplicates(["held_out", "horizon", "model", "logic", "level"])

    od = cfg["out_dir"]
    if not ind.empty: ind.to_csv(f"{od}/metrics_indomain.csv", index=False)
    if not dss.empty: dss.to_csv(f"{od}/dss_multicell.csv", index=False)
    if not reg.empty:
        reg[np.isclose(reg.threshold, cfg["regional_frac"])].to_csv(f"{od}/dss_regional.csv", index=False)
        reg.to_csv(f"{od}/dss_regional_thresholds.csv", index=False)
    if not dtr.empty:
        dtr.to_csv(f"{od}/dss_transfer_percell.csv", index=False)
        pool_transfer_dss(dtr).to_csv(f"{od}/dss_transfer_multicell.csv", index=False)
    if not tr.empty:  tr.to_csv(f"{od}/metrics_transfer.csv", index=False)
    if not ind.empty and not tr.empty:
        build_skill_map(meta, ind, tr, cfg).to_csv(f"{od}/skill_map.csv", index=False)
    print(f"[merge] indomain rows={len(ind)} transfer rows={len(tr)} dss={len(dss)} regional={len(reg)} "
          f"transfer-dss={len(dtr)} "
          f"-> {od}/ (metrics_indomain, metrics_transfer, dss_multicell, dss_regional, skill_map)")

# ------------------------------------ main ------------------------------------
def main(cfg=CONFIG_MC):
    os.makedirs(cfg["out_dir"], exist_ok=True)
    phase = cfg.get("phase", "all")
    sharding = (cfg.get("only_held") is not None) or (cfg.get("only_horizons") is not None)

    if phase == "merge":
        merge_shards(cfg); return

    meta, cells, cfg = prepare(cfg)
    print(f"Cells loaded: {list(cells.keys())}  |  TensorFlow: {P.HAS_TF}  |  phase={phase}  "
          f"only_held={cfg.get('only_held')}  horizons={_horizons(cfg)}")

    if phase in ("all", "indomain"):
        ind, dss, reg = run_indomain(cells, cfg)
        if sharding:
            tag = _htag(cfg); pd_ = _parts(cfg)
            ind.to_csv(f"{pd_}/indomain__{tag}.csv", index=False)
            dss.to_csv(f"{pd_}/dssmc__{tag}.csv", index=False)
            reg.to_csv(f"{pd_}/dssreg__{tag}.csv", index=False)
            print(f"[indomain shard] wrote parts/*__{tag}.csv")
        else:
            ind.to_csv(f"{cfg['out_dir']}/metrics_indomain.csv", index=False)
            dss.to_csv(f"{cfg['out_dir']}/dss_multicell.csv", index=False)
            reg[np.isclose(reg.threshold, cfg["regional_frac"])].to_csv(f"{cfg['out_dir']}/dss_regional.csv", index=False)
            reg.to_csv(f"{cfg['out_dir']}/dss_regional_thresholds.csv", index=False)
            print("\n=== POOLED DSS (OR, D1) ===")
            print(dss[(dss.logic == "or") & (dss.level == "D1")]
                  [["horizon", "model", "POD", "FAR", "CSI", "base_rate", "hits"]].round(3).to_string(index=False))
            print(f"\n=== REGIONAL >= {int(cfg['regional_frac']*100)}% ALERT (OR, D1) ===")
            print(reg[(reg.level == "D1") & np.isclose(reg.threshold, cfg["regional_frac"])]
                  [["horizon", "model", "POD", "FAR", "CSI", "base_rate", "hits"]]
                  .round(3).to_string(index=False))

    if phase in ("all", "transfer"):
        tr, dtr = run_transfer(cells, cfg)
        if sharding:
            tag = (cfg.get("only_held") or "allcells") + "__" + _htag(cfg); pd_ = _parts(cfg)
            tr.to_csv(f"{pd_}/transfer__{tag}.csv", index=False)
            dtr.to_csv(f"{pd_}/dsstr__{tag}.csv", index=False)
            print(f"[transfer shard] wrote parts/transfer__{tag}.csv and parts/dsstr__{tag}.csv")
        else:
            tr.to_csv(f"{cfg['out_dir']}/metrics_transfer.csv", index=False)
            dtr.to_csv(f"{cfg['out_dir']}/dss_transfer_percell.csv", index=False)
            pool_transfer_dss(dtr).to_csv(f"{cfg['out_dir']}/dss_transfer_multicell.csv", index=False)

    if phase == "all":
        smap = build_skill_map(meta, ind, tr, cfg)
        smap.to_csv(f"{cfg['out_dir']}/skill_map.csv", index=False)
        print(f"\n=== SKILL MAP (in-domain vs LOCO transfer) ===\n{smap.round(3).to_string(index=False)}")
        print(f"\nWrote canonical CSVs -> {cfg['out_dir']}/")
    elif sharding:
        print("Shard done. After all shards: run with phase='merge' to assemble final CSVs.")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile tcn_drought_rev6.py
"""tcn_drought_rev6.py  --  REV6: from forecast skill to warning skill (multi-cell)
================================================================================
REV6 adds four things to the REV5 multi-cell experiment. REV5 code and results are
left untouched (this module imports them); everything new is written to results_rev6/.

 1. Post-processing that restores the forecast distribution, fitted out of sample.
    A development model is trained on 1996-2016 and forecasts the validation years
    2017-2020; the post-processing is fitted on those validation forecasts and then
    applied to the test forecasts (2021-2025) of the final model (trained on 1996-2020,
    exactly as in REV5). The test period is never used for fitting.
      raw : the model output (as in REV5)
      QM  : empirical quantile mapping of each index (SPEI, SSI) onto its observed
            validation distribution
      VI  : variance inflation (mean and standard deviation matched to observation)
      THR : alert threshold shifted by a delta chosen on validation to maximise CSI
            (per model, lead, logic and level); the forecast itself is unchanged
    The same treatments are applied to persistence and climatology, so every method
    is compared on equal terms.

 2. Onset verification, where persistence cannot simply carry today's drought forward.
      onset (conditional): only issue days on which the observed state is BELOW the
            alert level; the event is that the level is reached at the target day.
            (Persistence can still alert here, but only through the seasonal change
            of the soil-moisture climatology between issue and target day.)
      episode: drought episodes (runs >= ep_min_dur days, gaps < ep_merge_gap merged);
            an observed onset is a hit if a forecast onset lies within +/- ep_tol days.
 3. Moving-block bootstrap (circular, blocks of boot_block days, same dates for every
    cell and method, i.e. paired) for CSI and for the CSI difference against persistence.
 4. Transformer with sinusoidal positional encoding and last-step + mean pooling
    (REV5 attention had no positional information). Reported as "Transformer".

Phases (CONFIG_R6["phase"]):
  "indomain" : train dev + final pooled models per lead, save values (resumable by lead)
  "transfer" : leave-one-cell-out for transfer_models (resumable by held-out cell / lead)
  "analyze"  : read the saved values and write every verification table (CPU, minutes)
  "all"      : indomain + transfer + analyze
Usage:  python tcn_drought_rev6.py      (edit CONFIG_R6 or pass a dict to main())
================================================================================
"""
import os, glob, json, gc
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
import tcn_drought_pipeline_v2 as P
import tcn_drought_multicell as M

CONFIG_R6 = dict(
    M.CONFIG_MC,
    out_dir="results_rev6",
    val_start="2017-01-01",                 # validation years used to fit the post-processing
    learned=["RandomForest", "TCN", "LSTM", "Transformer"],
    transfer_models=["TCN", "RandomForest"],
    postproc=("raw", "QM", "VI", "THR"),
    thr_grid=tuple(np.round(np.arange(-0.50, 1.501, 0.05), 2)),
    qm_quantiles=201,
    boot_B=1000, boot_block=60, boot_seed=2026,
    ep_min_dur=7, ep_merge_gap=10, ep_tol=7,
    only_cells=None,                        # testing only: restrict to a list of cell ids
    phase="indomain", only_held=None, only_horizons=None,
)

LEVEL_THR = np.array([-0.50, -0.84, -1.28, -1.64])           # USDM D0..D3, as P.DROUGHT_LEVELS
LNAME = {1: "D0", 2: "D1", 3: "D2", 4: "D3"}

# =============================== Transformer + PE ===============================
if P.HAS_TF:
    class SinusoidalPE(P.layers.Layer):
        """Adds the fixed sinusoidal position code of Vaswani et al. (2017)."""
        def build(self, input_shape):
            L, d = int(input_shape[1]), int(input_shape[2])
            pos = np.arange(L)[:, None]; i = np.arange(d)[None, :]
            ang = pos / np.power(10000.0, (2 * (i // 2)) / d)
            pe = np.where(i % 2 == 0, np.sin(ang), np.cos(ang)).astype("float32")
            self.pe = P.tf.constant(pe[None, :, :])
            super().build(input_shape)
        def call(self, x):
            return x + self.pe

    def build_transformer_pe(in_shape, out=2, d_model=64, heads=4):
        """REV5 encoder block (same width, heads, normalisation) + positional encoding;
        pooled by concatenating the last time step with the mean over the window."""
        L = P.layers
        inp = L.Input(shape=in_shape)
        x = SinusoidalPE()(L.Dense(d_model)(inp))
        a = L.MultiHeadAttention(num_heads=heads, key_dim=d_model)(x, x)
        x = L.LayerNormalization()(x + a)
        f = L.Dense(d_model, activation="relu")(x)
        x = L.LayerNormalization()(x + f)
        last = L.Lambda(lambda z: z[:, -1, :])(x)
        x = L.Concatenate()([last, L.GlobalAveragePooling1D()(x)])
        m = P.Model(inp, L.Dense(out)(x)); m.compile(optimizer="adam", loss="mse"); return m

    BUILDERS = {"TCN": P.build_tcn, "LSTM": P.build_lstm, "Transformer": build_transformer_pe}
else:
    BUILDERS = {}

# ================================ data / windows ================================
def prepare(cfg):
    meta, cells, cfg = M.prepare(cfg)
    if cfg.get("only_cells"):
        cells = {c: v for c, v in cells.items() if c in cfg["only_cells"]}
        meta = meta[meta.cell_id.isin(cfg["only_cells"])]
    dev = dict(cfg, val_end=cfg["train_end"], test_start=cfg["val_start"], test_end=cfg["val_end"])
    for cid in cells:                       # development split: scaler on <=2016, "test" = 2017-2020
        cells[cid]["d_dev"] = P.split_scale(cells[cid]["df"], dev)
    return meta, cells, cfg

def windows(d, zvec, h, cfg, split):
    Xd, Y = (d["Xtr"], d["Ytr"]) if split == "train" else (d["Xte"], d["Yte"])
    return P.build_windows(np.hstack([Xd, np.tile(zvec, (len(Xd), 1))]), Y, cfg["lookback"], h)

def fit_models(cells, ids, z, h, cfg, models, dkey, n_trees=None):
    """Pooled fit on the 'train' part of cells[cid][dkey] (dkey='d' final, 'd_dev' development).
    Neural models stop early on the last val_frac of each cell (REV5 rule)."""
    Xs, Ys = [], []
    for cid in ids:
        Xw, Yw, _ = windows(cells[cid][dkey], z[cid], h, cfg, "train"); Xs.append(Xw); Ys.append(Yw)
    fitted = {}
    if "RandomForest" in models:
        Xtr, Ytr = np.concatenate(Xs), np.concatenate(Ys)
        Xflat = Xtr.reshape(len(Xtr), -1)
        fitted["RandomForest"] = []
        for s in cfg["seeds"]:
            rf = RandomForestRegressor(n_estimators=n_trees or cfg["rf_trees"],
                                       max_features=cfg.get("rf_max_features", "sqrt"),
                                       random_state=s, n_jobs=-1)
            rf.fit(Xflat, Ytr); fitted["RandomForest"].append(("rf", rf))
        del Xtr, Xflat
    neural = [m for m in models if m in BUILDERS]
    if neural:
        Xf, Yf, Xv, Yv = M.split_per_cell(Xs, Ys, cfg["val_frac"])
        shape = (cfg["lookback"], Xf.shape[2])
        for m in neural:
            fitted[m] = [("keras", M.train_keras_val(BUILDERS[m], Xf, Yf, Xv, Yv, shape, cfg, s))
                         for s in cfg["seeds"]]
    return fitted

def forecast_frame(cells, cid, dkey, zvec, h, fitted, cfg):
    """One row per target day: observed indices at the target and issue day, and every
    method's forecast (GWETROOT, SPEI, SSI)."""
    gi, si = cfg["targets"].index("GWETROOT"), cfg["targets"].index("SPEI_1")
    d = cells[cid][dkey]; clim = cells[cid]["ssi_clim"]; L = cfg["lookback"]
    Xte, Yte, Ylast = windows(d, zvec, h, cfg, "test")
    inv = d["sY"].inverse_transform; Yo, Yl = inv(Yte), inv(Ylast)
    tdates = pd.DatetimeIndex(d["te_index"][L + h - 1: L + h - 1 + len(Yte)])
    idates = pd.DatetimeIndex(d["te_index"][L - 1: L - 1 + len(Yte)])
    tdoy, idoy = tdates.dayofyear.values, idates.dayofyear.values
    ssi = lambda g, doy: P.apply_ssi(g, doy, clim, cfg["spei_clip"])
    V = pd.DataFrame({"cell_id": cid, "date": tdates, "issue_date": idates,
                      "obs_gwet": Yo[:, gi], "obs_spei": Yo[:, si], "obs_ssi": ssi(Yo[:, gi], tdoy),
                      "iss_spei": Yl[:, si], "iss_ssi": ssi(Yl[:, gi], idoy)})
    fc = {"Persistence": Yl,
          "Climatology": np.column_stack([P.climatology_pred(d["tr_index"], d["tr_raw"][:, j], tdates)
                                          for j in range(Yo.shape[1])])}
    for m, mods in fitted.items():
        ps = [mm.predict(Xte.reshape(len(Xte), -1)) if kind == "rf" else mm.predict(Xte, verbose=0)
              for kind, mm in mods]
        fc[m] = inv(np.mean(ps, axis=0))
    for m, F in fc.items():
        V[f"{m}__gwet"] = F[:, gi]; V[f"{m}__spei"] = F[:, si]; V[f"{m}__ssi"] = ssi(F[:, gi], tdoy)
    return V

def methods_in(V):
    return [c[:-6] for c in V.columns if c.endswith("__spei")]

# ============================== post-processing ================================
def sev(x, shift=0.0):
    """Severity 0..4 (None, D0..D3) as P.severity, vectorised; NaN -> 0. `shift` (scalar or
    one value per row) moves every threshold up by delta (alert earlier) for THR."""
    x = np.asarray(x, float); shift = np.asarray(shift, float)
    thr = LEVEL_THR[None, :] + (shift[:, None] if shift.ndim else shift)
    return (x[:, None] <= thr).sum(1)

def status(spei, ssi, logic, shift=0.0):
    a, b = sev(spei, shift), sev(ssi, shift)
    return np.maximum(a, b) if logic == "or" else np.minimum(a, b)

def fit_qm(f, o, nq):
    q = np.linspace(0, 1, nq); ok = np.isfinite(f) & np.isfinite(o)
    fq, oq = np.quantile(f[ok], q), np.quantile(o[ok], q)
    fq = np.maximum.accumulate(fq) + np.arange(nq) * 1e-9               # strictly increasing
    return fq, oq

def apply_qm(x, par, clip):
    fq, oq = par; x = np.asarray(x, float); y = np.interp(x, fq, oq)
    lo, hi = x < fq[0], x > fq[-1]
    y[lo] = oq[0] + (x[lo] - fq[0]); y[hi] = oq[-1] + (x[hi] - fq[-1])
    return np.clip(y, -clip, clip)

def fit_vi(f, o):
    ok = np.isfinite(f) & np.isfinite(o)
    return float(f[ok].mean()), float(f[ok].std()), float(o[ok].mean()), float(o[ok].std())

def apply_vi(x, par, clip):
    mf, sf, mo, so = par
    return np.clip(mo + (np.asarray(x, float) - mf) * (so / sf if sf > 1e-9 else 1.0), -clip, clip)

def csi_counts(o, f):
    h = int((o & f).sum()); fa = int((~o & f).sum()); mi = int((o & ~f).sum())
    return h, fa, mi

def fit_postproc(Vval, cfg):
    """Fit QM / VI per method and index, and the THR delta per method, logic and level,
    on the validation forecasts of the development model. Returns (params, thr table)."""
    params, thr_rows = {}, []
    ostat = {lg: status(Vval.obs_spei.values, Vval.obs_ssi.values, lg) for lg in ("or", "and")}
    for m in methods_in(Vval):
        params[m] = {}
        for ix in ("spei", "ssi"):
            f, o = Vval[f"{m}__{ix}"].values, Vval[f"obs_{ix}"].values
            params[m][ix] = dict(QM=fit_qm(f, o, cfg["qm_quantiles"]), VI=fit_vi(f, o))
        for lg in ("or", "and"):
            for lev in cfg["levels"]:
                o = ostat[lg] >= lev; best = (-1.0, 0.0)
                for dl in cfg["thr_grid"]:
                    f = status(Vval[f"{m}__spei"].values, Vval[f"{m}__ssi"].values, lg, dl) >= lev
                    h, fa, mi = csi_counts(o, f); c = h / (h + fa + mi) if (h + fa + mi) else 0.0
                    if c > best[0] + 1e-12 or (abs(c - best[0]) <= 1e-12 and abs(dl) < abs(best[1])):
                        best = (c, float(dl))
                thr_rows.append(dict(model=m, logic=lg, level=LNAME[lev], delta=best[1], CSI_val=best[0]))
    return params, pd.DataFrame(thr_rows)

def add_postproc(V, params, cfg):
    clip = cfg["spei_clip"]
    for m, pm in params.items():
        for ix in ("spei", "ssi"):
            V[f"{m}__{ix}__QM"] = apply_qm(V[f"{m}__{ix}"].values, pm[ix]["QM"], clip)
            V[f"{m}__{ix}__VI"] = apply_vi(V[f"{m}__{ix}"].values, pm[ix]["VI"], clip)
    return V

def params_table(params):
    rows = []
    for m, pm in params.items():
        for ix in ("spei", "ssi"):
            mf, sf, mo, so = pm[ix]["VI"]
            rows.append(dict(model=m, index=ix, mean_fc=mf, sd_fc=sf, mean_obs=mo, sd_obs=so,
                             qm_fc_q05=float(np.interp(0.05, np.linspace(0, 1, len(pm[ix]["QM"][0])), pm[ix]["QM"][0])),
                             qm_obs_q05=float(np.interp(0.05, np.linspace(0, 1, len(pm[ix]["QM"][1])), pm[ix]["QM"][1]))))
    return pd.DataFrame(rows)

# ================================= training ====================================
def _free():
    gc.collect()
    if P.HAS_TF:
        P.tf.keras.backend.clear_session()

def _parts(cfg):
    p = os.path.join(cfg["out_dir"], "parts"); os.makedirs(p, exist_ok=True); return p

def _horizons(cfg):
    return cfg["only_horizons"] if cfg.get("only_horizons") else cfg["horizons"]

def continuous_rows(V, key):
    rows = []
    for cid, g in V.groupby("cell_id"):
        for m in methods_in(V):
            for tname, oc, pc, rc in [("GWETROOT", "obs_gwet", f"{m}__gwet", "Persistence__gwet"),
                                      ("SPEI_1", "obs_spei", f"{m}__spei", "Persistence__spei")]:
                rows.append(dict(**key, cell_id=cid, target=tname, model=m,
                                 **P.all_metrics(g[oc].values, g[pc].values, g[rc].values)))
    return rows

def run_indomain(cells, cfg):
    ids = list(cells.keys()); z = M.zscore_static(cells, ids, cfg)
    learned = [m for m in cfg["learned"] if m == "RandomForest" or m in BUILDERS]
    for h in _horizons(cfg):
        vfile = f"{_parts(cfg)}/values_indomain__h{h}.csv.gz"
        if os.path.exists(vfile):
            print(f"[indomain] h={h} already done"); continue
        fd = fit_models(cells, ids, z, h, cfg, learned, "d_dev")                     # 1996-2016
        Vval = pd.concat([forecast_frame(cells, c, "d_dev", z[c], h, fd, cfg) for c in ids])
        del fd; _free()
        params, thr = fit_postproc(Vval, cfg)
        ff = fit_models(cells, ids, z, h, cfg, learned, "d")                          # 1996-2020
        V = pd.concat([forecast_frame(cells, c, "d", z[c], h, ff, cfg) for c in ids])
        del ff; _free()
        V = add_postproc(V, params, cfg)
        key = dict(experiment="indomain", horizon=h)
        pd.DataFrame(continuous_rows(V, key)).to_csv(f"{_parts(cfg)}/cont_indomain__h{h}.csv", index=False)
        thr.assign(**key).to_csv(f"{_parts(cfg)}/thr_indomain__h{h}.csv", index=False)
        params_table(params).assign(**key).to_csv(f"{_parts(cfg)}/pp_indomain__h{h}.csv", index=False)
        add_postproc(Vval, params, cfg).to_csv(f"{_parts(cfg)}/valvalues_indomain__h{h}.csv.gz", index=False)
        V.to_csv(vfile, index=False)
        print(f"[indomain] h={h} done: {len(V)} target days x {V.cell_id.nunique()} cells", flush=True)

def run_transfer(cells, cfg):
    ids = list(cells.keys())
    held_list = [cfg["only_held"]] if cfg.get("only_held") else ids
    models = [m for m in cfg["transfer_models"] if m == "RandomForest" or m in BUILDERS]
    for held in held_list:
        train_ids = [c for c in ids if c != held]; z = M.zscore_static(cells, train_ids, cfg)
        for h in _horizons(cfg):
            vfile = f"{_parts(cfg)}/values_transfer__{held}__h{h}.csv.gz"
            if os.path.exists(vfile):
                print(f"[transfer] {held} h={h} already done"); continue
            nt = cfg.get("transfer_rf_trees")
            fd = fit_models(cells, train_ids, z, h, cfg, models, "d_dev", nt)
            # post-processing is fitted on the TRAINING cells only: the held-out cell stays unseen
            Vval = pd.concat([forecast_frame(cells, c, "d_dev", z[c], h, fd, cfg) for c in train_ids])
            del fd; _free()
            params, thr = fit_postproc(Vval, cfg)
            ff = fit_models(cells, train_ids, z, h, cfg, models, "d", nt)
            V = add_postproc(forecast_frame(cells, held, "d", z[held], h, ff, cfg), params, cfg)
            del ff; _free()
            key = dict(experiment="transfer", horizon=h)
            pd.DataFrame(continuous_rows(V, key)).assign(held_out=held).to_csv(
                f"{_parts(cfg)}/cont_transfer__{held}__h{h}.csv", index=False)
            thr.assign(**key, held_out=held).to_csv(f"{_parts(cfg)}/thr_transfer__{held}__h{h}.csv", index=False)
            V.to_csv(vfile, index=False)
            print(f"[transfer] held={held} h={h} done", flush=True)

# ================================= analysis ====================================
def variant(V, m, pp):
    """(spei, ssi) of method m under post-processing pp (THR uses the raw forecast)."""
    if pp in ("QM", "VI"):
        return V[f"{m}__spei__{pp}"].values, V[f"{m}__ssi__{pp}"].values
    return V[f"{m}__spei"].values, V[f"{m}__ssi"].values

def block_indices(n, B, block, seed):
    rng = np.random.default_rng(seed); nb = int(np.ceil(n / block))
    starts = rng.integers(0, n, size=(B, nb))
    idx = (starts[:, :, None] + np.arange(block)[None, None, :]) % n
    return idx.reshape(B, -1)[:, :n]

def per_date_counts(dates_inv, n_dates, o, f, mask=None):
    w = np.ones(len(o), bool) if mask is None else mask
    H = np.bincount(dates_inv, weights=(o & f & w).astype(float), minlength=n_dates)
    F = np.bincount(dates_inv, weights=(~o & f & w).astype(float), minlength=n_dates)
    Mi = np.bincount(dates_inv, weights=(o & ~f & w).astype(float), minlength=n_dates)
    return H, F, Mi

def scores(h, fa, mi, n=None):
    h, fa, mi = float(h), float(fa), float(mi)
    out = dict(hits=h, false_alarms=fa, misses=mi,
               POD=h / (h + mi) if (h + mi) else np.nan, FAR=fa / (h + fa) if (h + fa) else np.nan,
               CSI=h / (h + fa + mi) if (h + fa + mi) else np.nan,
               freq_bias=(h + fa) / (h + mi) if (h + mi) else np.nan)
    if n: out["base_rate"] = (h + mi) / n
    return out

def boot_csi(Hd, Fd, Md, idx):
    h, f, m = Hd[idx].sum(1), Fd[idx].sum(1), Md[idx].sum(1); den = h + f + m
    return np.where(den > 0, h / np.where(den > 0, den, 1), np.nan)

def runs(ev, merge_gap, min_dur):
    ev = np.asarray(ev, bool); idx = np.flatnonzero(ev)
    if len(idx) == 0: return []
    br = np.flatnonzero(np.diff(idx) > 1)
    st, en = np.r_[idx[0], idx[br + 1]], np.r_[idx[br], idx[-1]]
    merged = [[st[0], en[0]]]
    for s, e in zip(st[1:], en[1:]):
        if s - merged[-1][1] - 1 < merge_gap: merged[-1][1] = e
        else: merged.append([s, e])
    return [(s, e) for s, e in merged if e - s + 1 >= min_dur]

def match_onsets(obs_on, fc_on, tol):
    used, hits, dts = set(), 0, []
    for o in obs_on:
        cand = [(abs(f - o), j) for j, f in enumerate(fc_on) if j not in used and abs(f - o) <= tol]
        if cand:
            _, j = min(cand); used.add(j); hits += 1; dts.append(fc_on[j] - o)
    return hits, len(obs_on) - hits, len(fc_on) - len(used), dts

def thr_shift(V, thr, m, lg, lev):
    """Per-row THR delta: a constant from `thr` (one model fit) or, when the frame pools
    several held-out cells, each cell's own delta (thr has a held_out column)."""
    t = thr[(thr.model == m) & (thr.logic == lg) & (thr.level == LNAME[lev])]
    if len(t) == 0:
        return np.zeros(len(V))
    if "held_out" in t.columns and t.held_out.notna().any():
        return V.cell_id.map(dict(zip(t.held_out, t.delta))).fillna(0.0).values.astype(float)
    return np.full(len(V), float(t.delta.iloc[0]))

def episode_scores(V, o_ev, f_ev, cfg):
    hh = mm = ff = 0; dts = []
    for _, g in V.assign(_o=o_ev, _f=f_ev).groupby("cell_id", sort=False):
        oo = [s for s, _ in runs(g._o.values, cfg["ep_merge_gap"], cfg["ep_min_dur"]) if s > 0]
        fo = [s for s, _ in runs(g._f.values, cfg["ep_merge_gap"], cfg["ep_min_dur"]) if s > 0]
        a, b, c, d = match_onsets(oo, fo, cfg["ep_tol"]); hh += a; mm += b; ff += c; dts += d
    return dict(obs_episodes=hh + mm, fc_episodes=hh + ff, **scores(hh, ff, mm),
                timing_mean=float(np.mean(dts)) if dts else np.nan,
                timing_median=float(np.median(dts)) if dts else np.nan)

def analyze_frame(V, exp, h, thr, cfg, regional=True):
    """All verification for one experiment and lead. V holds one or many cells (sorted by
    cell and date inside). Bootstrap resamples blocks of target DATES, shared by all cells
    and methods (paired)."""
    V = V.sort_values(["cell_id", "date"]).reset_index(drop=True)
    udates, inv = np.unique(pd.to_datetime(V["date"]).values, return_inverse=True); nd = len(udates)
    bidx = block_indices(nd, cfg["boot_B"], cfg["boot_block"], cfg["boot_seed"] + h)
    methods = methods_in(V); pps = cfg["postproc"]
    out = dict(cont=[], day=[], onset=[], episode=[], regional=[])

    # continuous, index space: the price of post-processing in squared error, and dispersion
    for ix in ("spei", "ssi"):
        o = V[f"obs_{ix}"].values; ok = np.isfinite(o)
        ref = float(np.mean((o - V[f"Persistence__{ix}"].values)[ok] ** 2))
        for m in methods:
            for pp in ("raw", "QM", "VI"):
                f = V[f"{m}__{ix}" if pp == "raw" else f"{m}__{ix}__{pp}"].values; k = ok & np.isfinite(f)
                mse = float(np.mean((o[k] - f[k]) ** 2))
                out["cont"].append(dict(experiment=exp, horizon=h, index=ix, model=m, postproc=pp, MSE=mse,
                                        SS_vs_persist_raw=1 - mse / ref if ref > 0 else np.nan,
                                        sd_ratio=float(np.std(f[k]) / np.std(o[k]))))

    for lg in ("or", "and"):
        o_stat = status(V.obs_spei.values, V.obs_ssi.values, lg)
        i_stat = status(V.iss_spei.values, V.iss_ssi.values, lg)
        for lev in cfg["levels"]:
            o_ev = o_stat >= lev; below = i_stat < lev
            boot = {}
            for pp in pps:
                for m in methods:
                    sp, ss = variant(V, m, pp)
                    dl = thr_shift(V, thr, m, lg, lev) if pp == "THR" else np.zeros(len(V))
                    f_ev = status(sp, ss, lg, dl) >= lev
                    for scheme, mask in (("day", None), ("onset", below)):
                        Hd, Fd, Md = per_date_counts(inv, nd, o_ev, f_ev, mask)
                        n = int(len(o_ev) if mask is None else mask.sum())
                        row = dict(experiment=exp, horizon=h, model=m, postproc=pp, logic=lg, level=LNAME[lev],
                                   delta=float(np.mean(dl)), n=n, **scores(Hd.sum(), Fd.sum(), Md.sum(), n))
                        if lg == "or":
                            cb = boot_csi(Hd, Fd, Md, bidx)
                            row.update(CSI_lo=np.nanpercentile(cb, 2.5), CSI_hi=np.nanpercentile(cb, 97.5))
                            boot[(scheme, m, pp)] = (cb, row)
                        out[scheme].append(row)
                    if lg == "or":
                        out["episode"].append(dict(experiment=exp, horizon=h, model=m, postproc=pp,
                                                   level=LNAME[lev], delta=float(np.mean(dl)),
                                                   **episode_scores(V, o_ev, f_ev, cfg)))
            if lg == "or":          # paired bootstrap of CSI differences against persistence
                for (scheme, m, pp), (cb, row) in boot.items():
                    for tag, refk in (("vsPersRaw", (scheme, "Persistence", "raw")),
                                      ("vsPersSame", (scheme, "Persistence", pp))):
                        rb, rrow = boot[refk]; diff = np.nan_to_num(cb) - np.nan_to_num(rb)
                        row[f"dCSI_{tag}"] = np.nan_to_num(row["CSI"]) - np.nan_to_num(rrow["CSI"])
                        row[f"dCSI_{tag}_lo"] = float(np.percentile(diff, 2.5))
                        row[f"dCSI_{tag}_hi"] = float(np.percentile(diff, 97.5))
                        row[f"p_{tag}"] = float(np.mean(diff <= 0))     # one-sided: model not better

    # regional alert (in-domain: all cells forecast by the same pooled model)
    if regional and exp == "indomain" and V.cell_id.nunique() > 1:
        frac = cfg["regional_frac"]
        piv = lambda s: pd.Series(np.asarray(s), index=[V.date, V.cell_id]).unstack()
        for lev in cfg["levels"]:
            O = ((piv(status(V.obs_spei.values, V.obs_ssi.values, "or")) >= lev).mean(1) >= frac).values
            I = ((piv(status(V.iss_spei.values, V.iss_ssi.values, "or")) >= lev).mean(1) >= frac).values
            for pp in pps:
                for m in methods:
                    sp, ss = variant(V, m, pp)
                    dl = thr_shift(V, thr, m, "or", lev) if pp == "THR" else np.zeros(len(V))
                    Fr = ((piv(status(sp, ss, "or", dl)) >= lev).mean(1) >= frac).values
                    for scheme, w in (("day", np.ones(len(O), bool)), ("onset", ~I)):
                        out["regional"].append(dict(experiment=exp, horizon=h, model=m, postproc=pp, scheme=scheme,
                                                    level=LNAME[lev], threshold=frac, n=int(w.sum()),
                                                    **scores((O & Fr & w).sum(), (~O & Fr & w).sum(),
                                                             (O & ~Fr & w).sum(), int(w.sum()))))
    return {k: pd.DataFrame(v) for k, v in out.items()}

def analyze(cfg):
    od = cfg["out_dir"]; pd_ = os.path.join(od, "parts"); tabs = {}
    def collect(res, suffix=""):
        for k, v in res.items():
            if len(v): tabs.setdefault(k + suffix, []).append(v)
    hz = lambda f: int(f.split("__h")[-1].split(".")[0])
    for f in sorted(glob.glob(f"{pd_}/values_indomain__h*.csv.gz"), key=hz):
        h = hz(f); V = pd.read_csv(f, parse_dates=["date", "issue_date"])
        collect(analyze_frame(V, "indomain", h, pd.read_csv(f"{pd_}/thr_indomain__h{h}.csv"), cfg))
        print(f"[analyze] indomain h={h}", flush=True)
    tfiles = sorted(glob.glob(f"{pd_}/values_transfer__*__h*.csv.gz"))
    for h in sorted({hz(f) for f in tfiles}):
        fs = [f for f in tfiles if hz(f) == h]
        V = pd.concat([pd.read_csv(f, parse_dates=["date", "issue_date"]) for f in fs], ignore_index=True)
        thr = pd.concat([pd.read_csv(f.replace("values_", "thr_").replace(".csv.gz", ".csv")) for f in fs])
        collect(analyze_frame(V, "transfer", h, thr, cfg))                       # pooled over held-out cells
        for held, g in V.groupby("cell_id"):                                     # and per held-out cell
            collect({k: v.assign(held_out=held) for k, v in
                     analyze_frame(g, "transfer", h, thr[thr.held_out == held], cfg).items()
                     if k in ("day", "onset", "episode")}, "_percell")
        print(f"[analyze] transfer h={h} ({V.cell_id.nunique()} held-out cells)", flush=True)
    names = dict(cont="continuous_index_space", day="warning_day", onset="warning_onset",
                 episode="warning_episode", regional="warning_regional")
    for k, lst in tabs.items():
        base, _, suf = k.partition("_percell")
        fname = names[base] + ("_transfer_percell" if k.endswith("_percell") else "")
        pd.concat(lst, ignore_index=True).to_csv(f"{od}/{fname}.csv", index=False)
    for kind, fname in (("cont", "metrics_original_scale"), ("thr", "thr_deltas"), ("pp", "postproc_params")):
        fs = sorted(glob.glob(f"{pd_}/{kind}_*.csv"))
        if fs:
            pd.concat([pd.read_csv(f) for f in fs], ignore_index=True).to_csv(f"{od}/{fname}.csv", index=False)
    print(f"[analyze] wrote tables to {od}/")
    summary(cfg)

def summary(cfg):
    od = cfg["out_dir"]
    for name, scheme in (("warning_day.csv", "DAY-BASED"), ("warning_onset.csv", "ONSET (issue day below level)")):
        f = os.path.join(od, name)
        if not os.path.exists(f): continue
        d = pd.read_csv(f); d = d[(d.logic == "or") & (d.level == "D1")]
        for exp, g in d.groupby("experiment"):
            print(f"\n=== {exp}: {scheme}, OR, D1 -- CSI [95% block-bootstrap CI] ===")
            g = g.assign(cell=g.apply(lambda r: f"{r.CSI:.2f} [{r.CSI_lo:.2f},{r.CSI_hi:.2f}]", axis=1))
            print(g.pivot_table(index=["model", "postproc"], columns="horizon", values="cell", aggfunc="first")
                  .to_string())
    f = os.path.join(od, "warning_episode.csv")
    if os.path.exists(f):
        d = pd.read_csv(f); d = d[d.level == "D1"]
        for exp, g in d.groupby("experiment"):
            print(f"\n=== {exp}: EPISODE ONSETS, D1 -- CSI (hits/obs episodes) ===")
            g = g.assign(cell=g.apply(lambda r: f"{r.CSI:.2f} ({int(r.hits)}/{int(r.obs_episodes)})", axis=1))
            print(g.pivot_table(index=["model", "postproc"], columns="horizon", values="cell", aggfunc="first")
                  .to_string())

# =================================== main ======================================
def main(cfg=CONFIG_R6):
    cfg = dict(cfg); os.makedirs(cfg["out_dir"], exist_ok=True)
    phase = cfg.get("phase", "indomain")
    if phase in ("indomain", "transfer", "all"):
        meta, cells, cfg = prepare(cfg)
        print(f"REV6 | cells={list(cells)} | TF={P.HAS_TF} | phase={phase} | horizons={_horizons(cfg)} "
              f"| held={cfg.get('only_held')}")
        with open(os.path.join(cfg["out_dir"], "config_rev6.json"), "w") as fh:
            json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()
                       if isinstance(v, (str, int, float, list, tuple, type(None)))}, fh, indent=1)
        if phase in ("indomain", "all"): run_indomain(cells, cfg)
        if phase in ("transfer", "all"): run_transfer(cells, cfg)
    if phase in ("analyze", "all"):
        analyze(cfg)

if __name__ == "__main__":
    main()


In [ ]:
# 4 | Smoke test (about 1 min)
import importlib, numpy as np
import tcn_drought_pipeline_v2 as P, tcn_drought_multicell as M, tcn_drought_rev6 as R
for mod in (P, M, R): importlib.reload(mod)
Xs = [np.random.rand(200, 90, 13).astype('float32') for _ in range(3)]; Ys = [np.random.rand(200, 2).astype('float32') for _ in range(3)]
Xf, Yf, Xv, Yv = M.split_per_cell(Xs, Ys, 0.10)
cfg = dict(R.CONFIG_R6, epochs=1)
for name in ['TCN', 'LSTM', 'Transformer']:
    m = M.train_keras_val(R.BUILDERS[name], Xf, Yf, Xv, Yv, (90, 13), cfg, 0)
    print(f'  {name:12s} OK, params={m.count_params():,}')
x, y = np.random.randn(3000), np.random.randn(3000)
for lg in ('or', 'and'):
    assert (R.status(x, y, lg) == P.status_series(x, y, lg)).all()
print('Alert status identical to REV5: OK')


In [ ]:
# 5 | Configuration (leave as is; to split across two sessions set HORIZONS = [14, 1] and then [7, 30])
HORIZONS = [14, 1, 7, 30]
C = dict(R.CONFIG_R6); C.update(data_dir=DATA_DIR, out_dir=OUT_DIR)
print({k: C[k] for k in ['seeds', 'epochs', 'batch_size', 'rf_trees', 'val_frac', 'levels', 'postproc',
                          'boot_B', 'boot_block', 'ep_min_dur', 'ep_merge_gap', 'ep_tol']}, '| horizons', HORIZONS)


In [ ]:
# 6 | Train: development fit + final fit, one lead at a time (resumable within the session)
for h in HORIZONS:
    t = time.time(); R.main(dict(C, phase='indomain', only_horizons=[h]))
    print(f'>>> h={h} finished in {(time.time() - t) / 60:.1f} min', flush=True)


In [ ]:
# 7 | Verification tables (CPU, a few minutes)
R.main(dict(C, phase='analyze'))


In [ ]:
# 8 | Package for download
shutil.make_archive('/kaggle/working/results_rev6_indomain', 'zip', OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))


Download **`results_rev6_indomain.zip`** and send it back for checking and for the manuscript update.